[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_D.ipynb)


# Set D — From Passive → Spiking, Adaptation & Bursting — LEGO–Colab
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

A short passive recap, then spiking (threshold, f–I, refractory), **adaptation** surrogate, and **bursting‑like** behavior.

---
## Table of Contents
- [D0 Passive recap](#d0)
- [D1 Add HH & threshold search](#d1)
- [D2 gNa/gK toggles](#d2)
- [D3 Refractory (paired pulses)](#d3)
- [D4 f–I curve & rheobase](#d4)
- [D5 Waveform landmarks](#d5)
- [D6 Propagation teaser](#d6)
- [D7 Spike‑frequency adaptation (surrogate)](#d7)
- [D8 Quantify adaptation](#d8)
- [D9 Bursting‑like bouts](#d9)
- [D10 AHP/ADP notes](#d10)
- [D11 Phase‑plane glimpse](#d11)
- [D12 Numerical tips](#d12)

### Starter / Initialization

**Purpose**
Load the necessary Hodgkin-Huxley (HH) mechanisms, plotting tools, and specialized NEURON libraries. This step confirms the environment is properly configured for biophysical simulations.

---

### Post-Run Checklist
After executing the initialization code, record the following in your notes:
*   **NEURON Version:** Note the specific version string printed during import.
*   **Import Messages:** Document any specific warnings or "success" messages that appear.
*   **Initial Defaults:** List the starting values for $R_a$, $C_m$, the leak conductance, and the standard HH channel parameters (e.g., $g_{Na}$, $g_K$).

---

### Exercises

**Clamp Functionality**
State in one concise line: What does a current clamp specifically control, and what does it measure?
*   [Your Answer Here]

**Figure Requirements**
List three essential items that must appear in every figure generated in this notebook (e.g., units on axes, legend/labels, protocol parameters).
1.  
2.  
3.  

**Seed Control and Reproducibility**
Explain, in one or two sentences, why manual seed control is not required for these specific HH simulations but why it will become critical once noise or stochastic channel openings are introduced.
*   [Your Answer Here]

In [ ]:

!pip -q install neuron==8.2.4 >/dev/null 2>&1
try:
    from neuron import h
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np, matplotlib.pyplot as plt
print('Starter ready (NEURON expected).')


## D0 — Passive recap <a id='d0'></a>
### D0 — Passive recap

**Idea**: Verify your $\tau$ estimation procedure before engaging active conductances.

---

### What to change
Step amplitude and duration: Adjust the stimulus parameters.
Passive properties: Modify leak conductance ($g_{\text{pas}}$) and capacitance ($c_m$).

---

### Sanity checks
*   **Time Constant:** Mark $\tau$ at 63% of the step distance.
*   **Input Resistance:** Compute $R_{\text{in}} = \Delta V / \Delta I$ at steady state.
*   **Consistency:** Verify that $\tau \approx R_{\text{in}} \cdot C_m$ (check your units!).

---

### Predict → verify
*   **Capacitance:** $\uparrow c_m \rightarrow \uparrow \tau$.
*   **Conductance:** $\uparrow g_{\text{pas}} \rightarrow \downarrow R_{\text{in}}$ and $\downarrow \tau$.

---

### Exercises

**Capacitance Scaling**
Double $c_m$ and re-measure $\tau$. Report the new value.

**Conductance Scaling**
Halve $g_{\text{pas}}$ and report both the new $R_{\text{in}}$ and $\tau$.

**Conceptual Logic**
In one sentence: Why is it important to verify passive membrane baselines before introducing spiking conductances?
*   [Your Answer Here]


In [ ]:

# Passive recap: step response and τ
soma=h.Section(name='somaP'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('pas'); soma.g_pas=0.0002; soma.e_pas=-65
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=40; stim.amp=-0.05
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(60*ms)
plt.figure(figsize=(7,3.3)); plt.plot(t,v,'k'); plt.title('D0: Passive recap (τ via 63% rule)'); plt.tight_layout(); plt.show()


## D1 — Add HH & threshold search <a id='d1'></a>
### D1 — Add HH & threshold search

**Idea**
Move from passive to spiking dynamics. The goal is to determine the rheobase (the minimum current required to elicit a spike) and examine how the first-spike latency changes as a function of the injected current.

---

### What to change
Step amplitudes: Use a fine grid of current values near the suspected threshold to pinpoint the rheobase.
Duration: Ensure the pulse duration is sufficient for a spike to occur, especially at near-threshold currents.

---

### Checks
Spike detection: For each current amplitude, report "spike" or "no spike."
Latency recording: For every spike, record the first-spike latency (time from stimulus onset to the peak of the first action potential).
Visual verification: Plot the first-spike latency versus the injected current.

---

### Predict → verify
Latency decreases nonlinearly as the injected current exceeds the threshold. Observe the sharp decrease in latency just above rheobase.

---

### Exercises

**Threshold Table**
Build a small summary table with the following columns:
*   Injected Current (I)
*   Spike? (Yes/No)
*   First-spike Latency (ms)

**Rheobase Analysis**
Identify the smallest current (I) that successfully elicits a spike. 
*   **Explain:** Discuss how uncertainty or "jitter" arises when trying to determine a precise threshold near the rheobase in a computational or biological system.

**Qualitative Sketching**
Sketch (or describe) the qualitative curve of the first-spike latency versus the injected current (I) for values just above the threshold. Focus on the shape of the curve as it approaches the rheobase from the right.


In [ ]:

# D1 Add HH & threshold search
soma=h.Section(name='soma'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('pas'); soma.insert('hh')
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=20
for A in [0.03,0.05,0.07,0.09]:
    stim.amp=A
    v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65*mV); h.continuerun(35*ms)
    plt.figure(figsize=(6.4,2.4)); plt.plot(t,v,'k'); plt.title(f'D1: I={A} nA'); plt.tight_layout(); plt.show()


### **D2 — Ionic logic: Na⁺ upstroke vs. K⁺ downstroke**

**Idea**: The shape of an action potential is determined by the kinetic interplay between different ion channels. We can attribute specific spike phases to the **Na⁺ upstroke** and the **K⁺-dominated downstroke/AHP** by selectively reducing the maximum conductances ($g_{Na}$ or $g_K$).

---

### **What to change**
*   **Sodium Conductance:** Adjust the scale factor for `gnabar_hh`.
*   **Potassium Conductance:** Adjust the scale factor for `gkbar_hh`.

---

### **Sanity checks**
*   **Reduced $g_{Na}$:** Observe if higher current is required to reach threshold and if the spike upstroke (depolarization phase) becomes visibly slower.
*   **Reduced $g_K$:** Observe if the repolarization phase (downstroke) slows down and if the afterhyperpolarization (AHP) becomes shallower or disappears.

---

### **Predict → verify**
Show two overlaid plots for comparison:
1.  **Control vs. Low $g_{Na}$:** Use the same current injection for both to see how the sodium reduction affects the spike profile.
2.  **Control vs. Low $g_K$:** Use the same current injection for both to isolate the potassium-mediated effects on repolarization.

---

### **Exercises**

**Rheobase Sensitivity**
Reduce the maximum sodium conductance ($g_{Na}$) by **30%** and find the new rheobase (minimum current required to spike).
*   **Report:** Original rheobase vs. new rheobase.

**Action Potential Width**
Reduce the maximum potassium conductance ($g_K$) by **30%** and quantify the change in the action potential **half-width** (the duration of the spike at 50% of its peak amplitude).

**Mechanistic Conclusion**
In two sentences, explain how these specific conductance manipulations clarify the individual roles of Na⁺ and K⁺ ions in generating the action potential waveform.
*   [Your Answer Here]


### **D3 — Refractory (paired pulses)**

**Idea**
Separate absolute vs. relative refractoriness by using two identical pulses at varying Inter-Stimulus Intervals (ISIs). This reveals the time needed for ion channels (particularly Na⁺) to recover from inactivation.

---

### What to change
*   **ISI List:** Define a range of intervals between the first and second pulse (e.g., 1ms to 20ms).
*   **Pulse Parameters:** Use a pulse duration and amplitude that is slightly above the rheobase (supra-threshold).

---

### Sanity checks
*   **Success Tracking:** For each ISI, record whether the second pulse elicits a second action potential.
*   **Boundary Definition:** Use your data to define the **absolute refractory period** (no spike regardless of intensity) and the **relative refractory period** (spiking possible but harder to achieve).

---

### Predict → verify
Confirm that there is a minimum ISI below which no second spike can occur (absolute refractoriness) followed by a regime where the cell shows reduced excitability.

---

### Exercises

**Refractory Mapping**
Plot "Spike 2 Success" (using 0 for fail, 1 for success) vs. ISI. 
*   **Mark:** Indicate the boundary of the absolute refractory period on your plot.

**Excitability Recovery**
Choose a fixed ISI within the relative refractory range where the second spike initially fails. Gradually increase the amplitude of the second pulse until spiking returns.
*   **Report:** Note the required current increment compared to the first pulse.

**Rate Limiting**
In two sentences, explain how the existence of a refractory period sets a fundamental physical limit on a neuron's maximum firing rate.
*   [Your Answer Here]


## D4 — f–I curve & rheobase <a id='d4'></a>
### **D4 — f–I curve & rheobase**

**Idea**: Quantify the steady-state firing rate of the neuron as a function of the input current. This relationship, known as the **f–I curve**, defines the neuron's fundamental input-output gain.

---

### **What to change**
*   **Amplitude Range:** Sweep through a series of current levels, starting below threshold and increasing to high supra-threshold values.
*   **Step Duration:** Use long enough pulses to allow the firing rate to stabilize.
*   **Spike Detector:** Configure the detection threshold (e.g., counting 0 mV upward crossings).

---

### **Sanity checks**
*   **f–I Plot:** Plot the firing rate (in Hz) against the injected current (in nA).
*   **Rheobase:** Identify the minimum current required to elicit repetitive firing.
*   **Gain Analysis:** Approximate the slope of the curve in the linear regime just above the threshold.

---

### **Predict → verify**
Observe how the slope typically steepens as the current rises above the threshold and then begins to saturate at very high currents due to refractory period constraints.

---

### **Exercises**

**1. Variability and Reproducibility**
Produce an f–I curve that includes **error bars** generated from at least two repeated trials for each current amplitude.

**2. Gain Measurement**
Calculate the **gain** (Hz/nA) measured between two specific points on the linear portion of your curve. 
*   **Report:** Provide the slope and the current range used for the calculation.

**3. Model Comparison**
Note one specific reason why different types of models (e.g., reduced models like LIF versus biophysically detailed models like HH) might produce significantly different f–I slopes.
*   [Your Answer Here]


In [ ]:

# D4 f–I curve (rough)
soma=h.Section(name='somaFI'); soma.L=soma.diam=20; soma.insert('hh')
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=300
amps=np.linspace(0.04,0.15,8); rates=[]
for A in amps:
    stim.amp=A
    v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65*mV); h.continuerun(320*ms)
    import numpy as np
    vv=np.array(v); cross=(vv[1:]>=0)&(vv[:-1]<0); spikes=cross.sum(); rates.append(1000*spikes/300.0)
plt.figure(figsize=(5.4,3.2)); plt.plot(amps,rates,'o-'); plt.xlabel('nA'); plt.ylabel('Hz'); plt.title('D4: f–I'); plt.tight_layout(); plt.show()


### D5 — Waveform landmarks

**Idea**
Link specific action potential (AP) waveform features to their underlying ionic mechanisms by measuring key landmarks: peak amplitude, half-width, afterhyperpolarization (AHP) depth, and the maximum rate of rise ($dV/dt$) during the upstroke.

---

### What to change
Measurement window: Define the time range to capture the full AP and AHP.
Detection thresholds: Set criteria for identifying the spike peak and the start/end of the upstroke.

---

### Sanity checks
Landmark Table: Report a small table for a single current step containing:
*   Peak Amplitude (mV)
*   Half-width (ms)
*   AHP Depth (mV)
*   Maximum $dV/dt$ (V/s)

---

### Exercises

**Current-Dependent Width**
Measure and compare the AP half-width at two different current intensities: one near the rheobase and one significantly above it.
*   **Report:** Observed half-widths and any visible differences.

**AHP and Recovery**
Discuss, in two or three sentences, how the depth and duration of the AHP might provide a "preview" of the neuron's adaptation or recovery dynamics for subsequent spikes.
*   [Your Answer Here]

### **D6 — Propagation teaser**

**Idea**: Preview axial coupling: a longer or connected process (like an axon or dendrite) shows conduction of the spike waveform. This setup demonstrates how a signal travels spatially with a characteristic delay and minor attenuation.

---

### What to change
Cable length and diameter: Experiment with the physical dimensions of the process.
Recording sites: Place recording electrodes at various distances from the somatic injection site.

---

### Sanity checks
Trace Comparison: Plot the somatic and distal voltage traces on the same axes.
Latency Annotation: Annotate the time lag between the peak of the spike at the soma and its arrival at the distal recording site.

---

### Exercises

**Axial Influence**
Increase the axial resistance ($R_a$) and observe the resulting changes in conduction velocity and signal shape.
*   **Report:** How does higher $R_a$ affect the timing and amplitude of the distal spike?

**Interpretation Caveat**
State one significant caveat about interpreting "propagation" results when using a single-compartment surrogate or a simplified cable model compared to a real biological axon.
*   [Your Answer Here]


## D7 — Spike‑frequency adaptation (surrogate) <a id='d7'></a>
### D7 — Spike-frequency adaptation (surrogate)

**Idea**
Emulate adaptation with a slow inhibitory conductance superimposed on HH spiking. This approach demonstrates how a slow negative feedback mechanism—similar to a calcium-activated potassium current ($I_{AHP}$)—causes the interval between spikes to lengthen over time.

---

### What to change
*   **Adaptation Dynamics:** Onset time, time constant ($\tau$), and maximum conductance ($g_{max}$) of the slow inhibitory current.
*   **Stimulus:** Step amplitude of the injected current.

---

### Sanity checks
*   **Interval Analysis:** Compute the first inter-spike interval ($\text{ISI}_1$) and compare it to the fifth ($\text{ISI}_5$) or the final interval in the train.
*   **Adaptation Metric:** Report the ratio of the steady-state (last) firing rate to the initial (first) firing rate.

---

### Predict → verify
Increasing the maximum conductance ($g_{max}$) or the time constant of the slow inhibition will strengthen the adaptation, resulting in a more pronounced increase in inter-spike intervals over the duration of the stimulus.

---

### Exercises

**Adaptation Sensitivity Sweep**
Sweep the maximum conductance ($g_{max}$) of the slow-inhibition. Plot the "last/first rate ratio" versus $g_{max}$. 
*   **Observe:** At what $g_{max}$ does the neuron reach a steady-state firing rate of zero (complete cessation of firing)?

**Temporal Dynamics**
Keep $g_{max}$ fixed and double the inhibitory time constant. 
*   **Comment:** Describe how the sequence of ISIs changes compared to your baseline simulation.

**Network Implications**
In two sentences, explain how spike-frequency adaptation changes a neuron’s functional role when it is part of a larger recurrent network.
*   [Your Answer Here]


In [ ]:

# D7 Adaptation surrogate via slow inhibitory conductance
soma=h.Section(name='somaSFA'); soma.L=soma.diam=20; soma.insert('hh')
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=1000; stim.amp=0.12
synI=h.AlphaSynapse(soma(0.5)); synI.onset=100; synI.tau=200; synI.gmax=0.002; synI.e=-70
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(1100*ms)
plt.figure(figsize=(7,3.3)); plt.plot(t,v,'k'); plt.title('D7: Adaptation surrogate'); plt.tight_layout(); plt.show()


### **D8 — Quantify adaptation**

**Idea**
Formalize metrics for spike-frequency adaptation to characterize how a neuron's excitability changes over time. Common methods include calculating an **adaptation index** ($\text{ISI}_{\text{last}} / \text{ISI}_{\text{first}}$) or measuring the slope of inter-spike intervals (ISIs) across the duration of a stimulus.

---

### What to change
*   **Metric definition:** Choose between a simple ratio index or a linear regression slope of ISIs vs. time.
*   **Windowing:** Adjust which spikes are included in the "first" and "last" calculations (e.g., first 2 vs. last 2, or first 2 vs. last 5).

---

### Sanity checks
*   **Scalar Index:** Provide a calculated adaptation index for your current simulation.
*   **Visual Trend:** Generate a plot showing **ISI vs. Spike Number** to confirm the expected monotonic increase.

---

### Exercises

**Comparative Metrics**
Set up two different adaptation configurations (using the surrogate inhibition from D7) that result in the same **mean firing rate** but yield significantly different **adaptation indices**. 
*   **Report:** The parameters used and the resulting indices for both cases.

**Lab Assessment Logic**
If you were grading a lab report, justify which specific metric (index vs. slope) you would prefer to see and why. Consider factors like ease of calculation, robustness to noise, and biological relevance.
*   [Your Answer Here]


## D9 — Bursting‑like bouts <a id='d9'></a>

### **D9 — Bursting-like bouts**

**Idea**
Create bursts by modulating the input with a slow envelope atop standard Hodgkin-Huxley (HH) spiking. This allows for the identification of burst onsets and inter-burst intervals (IBIs) to characterize non-tonic firing patterns.

---

### **What to change**
*   **Envelope Dynamics:** Adjust the amplitude and frequency of the slow driving envelope.
*   **Base Excitability:** Modify the base depolarization level to shift the neuron between tonic and bursting regimes.

---

### **Sanity checks**
*   **Burst Detection:** Use a threshold on the short-term firing rate or a minimum "spikes-per-window" count to detect distinct bursts.
*   **Metric Summary:** Summarize the total burst count, the number of spikes per burst, and the inter-burst intervals (IBIs).

---

### **Predict → verify**
*   **Amplitude:** Increasing the envelope amplitude should increase the number of spikes per burst.
*   **Frequency:** Slowing the envelope frequency should lengthen the inter-burst intervals.

---

### **Exercises**

**Visualization and Alignment**
Produce a combined plot featuring a **burst raster** (individual spike times) and an overlaid **slow envelope** trace to visually confirm the alignment of bursts with the peaks of the driving signal.

**Biophysical Mechanisms**
State two specific biophysical "levers" (conductances or currents) that could realize a similar bursting effect in a real neuron without an external envelope.
*   *Example 1: A slow, persistent inward current (like $I_{NaP}$ or $I_{Ca,L}$).*
*   *Example 2: A calcium-dependent potassium current ($I_{AHP}$) providing slow negative feedback.*

***

**Next Step:** Would you like the **Python code** to generate this sinusoidal envelope and perform the burst detection analysis?


In [ ]:

# D9 Bursting-like bouts via slow envelope
soma=h.Section(name='somaBurst'); soma.L=soma.diam=20; soma.insert('hh')
ic=h.IClamp(soma(0.5)); ic.delay=0; ic.dur=1e9; ic.amp=0.0
T=4000; import numpy as np
tt=np.arange(0,T,0.1); envelope=0.07+0.05*np.sin(2*np.pi*tt/1500.0)
ivec=h.Vector(envelope); tvec=h.Vector(tt)
ivec.play(ic._ref_amp,tvec,1)
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(T*ms)
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3.3)); plt.plot(t,v,'k'); plt.title('D9: Bursting-like bouts'); plt.tight_layout(); plt.show()


### D10 — AHP/ADP notes

**Idea**
Qualitatively relate afterhyperpolarization (AHP) and afterdepolarization (ADP) to expected changes in excitability between spikes. These post-spike potentials shape the neuron's "readiness" to fire again by shifting the membrane potential relative to the threshold.

---

### What to change
None (This is a conceptual note block for your observations).

---

### Exercises

**AHP Hypothesis**
Provide one specific physiological or biophysical hypothesis (e.g., changing a specific conductance or ion concentration) that would lead to an increase in **AHP depth**.
*   [Your Answer Here]

**Refractory Consequences**
Based on your understanding of AHP and ADP, predict the consequences for **paired-pulse thresholds**. How would a deeper AHP or a prominent ADP specifically change the amount of current needed for a second pulse to reach threshold at a short interval?
*   [Your Answer Here]

***

### D11 — Phase-plane glimpse

**Idea**
Sketch the logic of spike initiation in a 2-variable portrait (typically voltage $V$ and a recovery variable $w$ or $n$). This geometric approach allows us to visualize how the system transitions from a stable rest state to a limit cycle or an action potential trajectory.

---

### Exercises

**Nullclines and Fixed Points**
In two sentences, define **nullclines** and describe what is represented by their intersection point.
*   [Your Answer Here]

**Threshold Manifolds**
Explain qualitatively how a trajectory (the state of the neuron) crosses a **threshold manifold** to initiate a spike, rather than returning to the resting equilibrium.
*   [Your Answer Here]

***

### D12 — Numerical tips

**Idea**
Ensure simulation accuracy and reliability by stabilizing numerical methods (e.g., adjusting step size), detecting spikes robustly (e.g., using 0 mV crossings with hysteresis), and labeling figures for consistent reproducibility.

---

### Exercises

**Step Size and Tolerance**
Run a standard simulation and then halve the maximum time step ($dt$). 
*   Show that the results are preserved within a small tolerance.
*   Report the relative **runtime cost** (increase in computation time) associated with this finer resolution.

**Hysteretic Spike Detection**
Implement a **hysteretic spike detector** (e.g., triggers at 0 mV, resets at -20 mV).
*   Compare the spike counts from this method to a "naïve" threshold detector that only uses a single voltage value. 
*   Discuss why hysteresis is often necessary in noisy or complex simulations.


## Reflection
- How do the passive τ expectations shape spike latency and f–I slope?
- Which adaptation/bursting parameters best serve as **control knobs** for networks in later sets?


---

## Practice / Discussion Questions — Set D — Spiking, Adaptation, Bursting

1) State how voltage‑gated Na⁺ and K⁺ currents produce a spike’s upstroke and downstroke.
2) *Explain*: What is **absolute vs relative refractoriness** at the channel level?
3) Describe one mechanism for **spike‑frequency adaptation** and its effect on ISIs.
4) Define **bursting** and name a minimal set of variables that can support it.
5) When is a **reduced model** preferable to HH‑type dynamics for teaching/exploration?
6) *Predict*: How will adding a slow outward current change the ISI sequence to a step input?
7) Describe the **phase‑plane (qualitative)** idea behind spike initiation in a 2‑variable model.
8) Compare two **adaptation** mechanisms (e.g., M‑current vs Ca²⁺‑dependent K⁺).
9) *Critique*: One pitfall of teaching with an overly simple spike model.
10) Explain how spike threshold can **vary** with recent membrane history.
11) Why might a neuron fail to spike to inputs that spiked it earlier (without parameter changes)?
12) *Scenario*: Two cells receive identical barrages; one bursts, one adapts. Suggest a minimal ionic difference.
13) A good **metric** to quantify adaptation in a step‑depolarization protocol.
14) One reason to add noise to spike models and one reason not to (for education).
15) Outline a **spike‑count vs input** experiment and what would indicate adaptation.
16) Explain “**dynamic threshold**” in intuitive terms.
17) *Design*: Two‑condition step protocol to elicit bursting and list expected traces.
18) Two **biophysical levers** that can turn adapting into bursting.
19) How does adding adaptation change a neuron’s role in a **network**?
20) Summarize how Set D concepts inform **STM** and **WTA** behavior.
